In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

In [3]:
df = pd.read_csv("C:/Users/admin/Desktop/YRES/Andi Dong Submission -- Task 5/Weekly_sign-up_report_dashboard_1775825037.xlsx - weekly-sign-up-report.csv", dtype = str)

In [4]:
df = df.replace(r'^\s*$', np.nan, regex=True)

In [5]:
df = df.dropna(how='all').copy()

In [6]:
header_signature = list(df.columns)

In [7]:
mask_repeated_header = df.apply(lambda row: list(row.values) == header_signature, axis = 1)

In [8]:
df = df.loc[~mask_repeated_header].copy()

In [9]:
rename_map = {'What type of placement are you interested in?': 'Placement type',
             'Which province do you reside in?': 'Province',
             'What City do you live in?': 'City',
             'Which organization referred you to this opportunity?': 'Referral',
    'What is your age range?': 'Age range'}
df = df.rename(columns=rename_map)

In [11]:
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

In [12]:
df = df.replace({'':np.nan})

In [13]:
df['City'].unique()

array(['Toronto', 'Markham', 'Ottawa', 'Richmond Hill', 'Stouffville',
       'Surrey, Metro Vancouver', 'Thornhill', 'Vaughan', 'Woodbridge',
       'markham', 'Mississauga', 'Etobicoke', 'Ajax', 'Coquitlam',
       'Scarborough', 'East Gwillimbury'], dtype=object)

In [19]:
df['City'] = df['City'].str.strip().str.title()

In [20]:
df['City']

0                     Toronto
1                     Markham
2                     Markham
3                     Toronto
4                     Markham
5                      Ottawa
6               Richmond Hill
7                 Stouffville
8                     Toronto
9                     Markham
10                    Toronto
11    Surrey, Metro Vancouver
12                  Thornhill
13                    Vaughan
14                     Ottawa
15                    Vaughan
16                    Toronto
17                    Toronto
18                    Toronto
19                 Woodbridge
20                    Markham
21                Mississauga
22                    Markham
23                  Etobicoke
24                       Ajax
25                    Toronto
26                  Coquitlam
27                     Ottawa
28                Scarborough
29                     Ottawa
30                Mississauga
31                  Etobicoke
32                    Markham
33        

In [21]:
df['City'].unique()

array(['Toronto', 'Markham', 'Ottawa', 'Richmond Hill', 'Stouffville',
       'Surrey, Metro Vancouver', 'Thornhill', 'Vaughan', 'Woodbridge',
       'Mississauga', 'Etobicoke', 'Ajax', 'Coquitlam', 'Scarborough',
       'East Gwillimbury'], dtype=object)

In [22]:
# 处理特殊的逗号情况，例如将 "Surrey, Metro Vancouver" 简化为 "Surrey"
df['City'] = df['City'].str.split(',').str[0]

# 再次查看结果
print(df['City'].unique())

['Toronto' 'Markham' 'Ottawa' 'Richmond Hill' 'Stouffville' 'Surrey'
 'Thornhill' 'Vaughan' 'Woodbridge' 'Mississauga' 'Etobicoke' 'Ajax'
 'Coquitlam' 'Scarborough' 'East Gwillimbury']


In [23]:
df['City']

0              Toronto
1              Markham
2              Markham
3              Toronto
4              Markham
5               Ottawa
6        Richmond Hill
7          Stouffville
8              Toronto
9              Markham
10             Toronto
11              Surrey
12           Thornhill
13             Vaughan
14              Ottawa
15             Vaughan
16             Toronto
17             Toronto
18             Toronto
19          Woodbridge
20             Markham
21         Mississauga
22             Markham
23           Etobicoke
24                Ajax
25             Toronto
26           Coquitlam
27              Ottawa
28         Scarborough
29              Ottawa
30         Mississauga
31           Etobicoke
32             Markham
33       Richmond Hill
34    East Gwillimbury
35             Markham
36             Toronto
Name: City, dtype: object

In [24]:
df['Province'].unique()

array(['Ontario', 'British Columbia'], dtype=object)

In [25]:
df['City'].isna().sum()

np.int64(0)

In [26]:
df['Province'].isna().sum()

np.int64(0)

In [27]:
# 本次City和Province无异常空值结果，TASK 4的help guide直接skip！

In [28]:
standard_provinces = {
    'Ontario', 'Quebec', 'British Columbia', 'Alberta', 'Manitoba', 
    'Saskatchewan', 'Nova Scotia', 'New Brunswick', 'Newfoundland and Labrador', 
    'Prince Edward Island', 'Northwest Territories', 'Yukon', 'Nunavut'
}

In [29]:
def infer_country(row):
    province = row['Province']
    city = row['City']
    if province in standard_provinces:
        return 'Canada'
    elif province == 'Outside Canada':
        return 'International'
    else:
        return None

df['Country'] = df.apply(infer_country, axis = 1)
    

In [30]:
df

,Item ID,Creation log,Placement type,Province,City,Referral,Age range,Country
0,11667100467,"Kevin Apr 3, 2026 12:15 AM",Flexible - I can do any format,Ontario,Toronto,Volunteer Connect York Region,15 - 19,Canada
1,11670946034,"Kevin Apr 3, 2026 3:17 PM",100% In-Person,Ontario,Markham,Volunteer Connect York Region,15 - 19,Canada
2,11671573914,"Kevin Apr 3, 2026 4:55 PM",100% Virtual,Ontario,Markham,School Guidance,12 - 14,Canada
3,11672548392,"Kevin Apr 3, 2026 8:50 PM",Flexible - I can do any format,Ontario,Toronto,Parents,12 - 14,Canada
4,11672693905,"Kevin Apr 3, 2026 10:10 PM",100% In-Person,Ontario,Markham,School Guidance,15 - 19,Canada
5,11673946009,"Kevin Apr 4, 2026 2:00 PM",100% In-Person,Ontario,Ottawa,Parents,12 - 14,Canada
6,11674114631,"Kevin Apr 4, 2026 4:14 PM","Some virtual, some in-person",Ontario,Richmond Hill,Parents,15 - 19,Canada
7,11674692755,"Kevin Apr 5, 2026 1:10 AM",Flexible - I can do any format,Ontario,Stouffville,Parents,12 - 14,Canada
8,11675611939,"Kevin Apr 5, 2026 12:46 PM",100% In-Person,Ontario,Toronto,Volunteer Toronto,15 - 19,Canada
9,11675684623,"Kevin Apr 5, 2026 1:53 PM",100% In-Person,Ontario,Markham,School Guidance,15 - 19,Canada


In [31]:
df[df['Province'] == 'Other province']

,Item ID,Creation log,Placement type,Province,City,Referral,Age range,Country


In [32]:
df[df['Country'] == 'International']

,Item ID,Creation log,Placement type,Province,City,Referral,Age range,Country


In [33]:
# 本周无需额外处理！

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37 entries, 0 to 36
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Item ID         37 non-null     object
 1   Creation log    37 non-null     object
 2   Placement type  37 non-null     object
 3   Province        37 non-null     object
 4   City            37 non-null     object
 5   Referral        37 non-null     object
 6   Age range       37 non-null     object
 7   Country         37 non-null     object
dtypes: object(8)
memory usage: 2.6+ KB


In [35]:
# 以上代表所有instruction指标的task任务完成！接下来更改格式方便去做可视化
def parse_creation_log(text):
    if pd.isna(text):
        return pd.NaT

    cleaned = re.sub(r'^[A-Za-z]+\s+', '', str(text))
    
    try:
        return datetime.strptime(cleaned, "%b %d, %Y %I:%M %p")
    except:
        return pd.NaT

df['Created Datetime'] = df['Creation log'].apply(parse_creation_log)
df['Created Date'] = df['Created Datetime'].dt.date
df['Created Weekday'] = df['Created Datetime'].dt.day_name()

In [36]:
df

,Item ID,Creation log,Placement type,Province,City,Referral,Age range,Country,Created Datetime,Created Date,Created Weekday
0,11667100467,"Kevin Apr 3, 2026 12:15 AM",Flexible - I can do any format,Ontario,Toronto,Volunteer Connect York Region,15 - 19,Canada,2026-04-03 00:15:00,2026-04-03,Friday
1,11670946034,"Kevin Apr 3, 2026 3:17 PM",100% In-Person,Ontario,Markham,Volunteer Connect York Region,15 - 19,Canada,2026-04-03 15:17:00,2026-04-03,Friday
2,11671573914,"Kevin Apr 3, 2026 4:55 PM",100% Virtual,Ontario,Markham,School Guidance,12 - 14,Canada,2026-04-03 16:55:00,2026-04-03,Friday
3,11672548392,"Kevin Apr 3, 2026 8:50 PM",Flexible - I can do any format,Ontario,Toronto,Parents,12 - 14,Canada,2026-04-03 20:50:00,2026-04-03,Friday
4,11672693905,"Kevin Apr 3, 2026 10:10 PM",100% In-Person,Ontario,Markham,School Guidance,15 - 19,Canada,2026-04-03 22:10:00,2026-04-03,Friday
5,11673946009,"Kevin Apr 4, 2026 2:00 PM",100% In-Person,Ontario,Ottawa,Parents,12 - 14,Canada,2026-04-04 14:00:00,2026-04-04,Saturday
6,11674114631,"Kevin Apr 4, 2026 4:14 PM","Some virtual, some in-person",Ontario,Richmond Hill,Parents,15 - 19,Canada,2026-04-04 16:14:00,2026-04-04,Saturday
7,11674692755,"Kevin Apr 5, 2026 1:10 AM",Flexible - I can do any format,Ontario,Stouffville,Parents,12 - 14,Canada,2026-04-05 01:10:00,2026-04-05,Sunday
8,11675611939,"Kevin Apr 5, 2026 12:46 PM",100% In-Person,Ontario,Toronto,Volunteer Toronto,15 - 19,Canada,2026-04-05 12:46:00,2026-04-05,Sunday
9,11675684623,"Kevin Apr 5, 2026 1:53 PM",100% In-Person,Ontario,Markham,School Guidance,15 - 19,Canada,2026-04-05 13:53:00,2026-04-05,Sunday


In [37]:
df['Age range'].info()

<class 'pandas.core.series.Series'>
Index: 37 entries, 0 to 36
Series name: Age range
Non-Null Count  Dtype 
--------------  ----- 
37 non-null     object
dtypes: object(1)
memory usage: 592.0+ bytes


In [38]:
df.to_csv("YRES_TASK_5_Weekly_signup_report_cleaned_data.csv", index=False, encoding="utf-8")